# Pipeline Validation

This notebook validates that the modular implementation in `src/`
produces the same cell detections as the original notebook pipeline.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import sys

# Add project root to Python path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from pathlib import Path

In [ ]:
PROJECT_ROOT = Path.cwd().parent

DATA_ROOT = PROJECT_ROOT / "data/sample"

SAMPLE_ID = "44b6_0113de3b"

SAMPLE_PATH = DATA_ROOT / "biohub_5samples_20timepoints" / "train" / f"{SAMPLE_ID}" / f"{SAMPLE_ID}.zarr"

OUTPUT_DIR = (
        DATA_ROOT
        / "processed"
        / "stage_6_processed_dataset"
        / SAMPLE_ID
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [ ]:
from tqdm.auto import tqdm
import numpy as np

from src.io.zarr_loader import open_sample, load_timepoint
from src.preprocessing.pipeline import preprocess_volume
from src.masking.pipeline import create_binary_mask
from src.segmentation.pipeline import segment_instances
from src.detection.pipeline import detect_cells
from src.feature_extraction.features import extract_cell_features


# Open the sample once to determine the number of timepoints
sample = open_sample(SAMPLE_PATH)
num_timepoints = sample.shape[0]

print(f"Processing {num_timepoints} timepoints...")

# Create output directories
preprocessing_dir = OUTPUT_DIR / "preprocessing"
masking_dir = OUTPUT_DIR / "masking"
segmentation_dir = OUTPUT_DIR / "segmentation"
cells_dir = OUTPUT_DIR / "cells"

for directory in (
        preprocessing_dir,
        masking_dir,
        segmentation_dir,
        cells_dir
):
    directory.mkdir(parents=True, exist_ok=True)


# Process every timepoint
for t in tqdm(range(num_timepoints), desc=SAMPLE_ID):

    # ------------------------
    # Stage 1
    # ------------------------
    volume = load_timepoint(SAMPLE_PATH, t)

    # ------------------------
    # Stage 2
    # ------------------------
    processed = preprocess_volume(volume)

    # ------------------------
    # Stage 3
    # ------------------------
    binary_mask = create_binary_mask(processed)

    # ------------------------
    # Stage 4
    # ------------------------
    instance_labels = segment_instances(binary_mask)

    # ------------------------
    # Stage 5
    # ------------------------
    detected_cells = detect_cells(instance_labels)

    # ------------------------
    # Stage 6
    # ------------------------
    cells = extract_cell_features(
        cells_df=detected_cells,
        labels=instance_labels,
        volume=volume,
    )

    # ------------------------
    # Save Outputs
    # ------------------------

    np.save(
        preprocessing_dir / f"t{t:03d}.npy",
        processed,
    )

    np.save(
        masking_dir / f"t{t:03d}.npy",
        binary_mask,
    )

    np.save(
        segmentation_dir / f"t{t:03d}.npy",
        instance_labels,
    )

    cells.to_csv(
        cells_dir / f"t{t:03d}.csv",
        index=False,
    )

print("✅ Processing complete.")